# StreamGuard: Real-Time Fraud Engine under Delayed Feedback & Concept Drift
**Author:** Rangga  
**Repository:** [streamguard-fraud-engine](https://github.com/Rangga11268/streamguard-fraud-engine)  
**Architecture:** Streaming Feature Store, Dual-Memory Delayed Feedback, Cost-Matrix Routing & Unsupervised PSI Drift

---

### Core Mathematical & Engineering Formulations

Financial fraud systems operate under two unforgiving conditions:
1. **Verification Latency**: Ground-truth chargebacks arrive 14 to 30 days after transaction settlement.
2. **Extreme Class Imbalance**: Genuine fraud accounts for $<0.1\%$ of transaction volume.

#### 1. Unsupervised Population Stability Index (PSI)
To detect covariate shift and concept drift before delayed labels mature, StreamGuard computes continuous PSI over streaming probability outputs:
$$PSI = \sum_{b=1}^B \left( P_{curr}(b) - P_{ref}(b) \right) \cdot \ln\left( \frac{P_{curr}(b) + \epsilon}{P_{ref}(b) + \epsilon} \right)$$
- $PSI < 0.10$: Stationary distribution (No intervention required).
- $0.10 \le PSI < 0.25$: Covariate warning (Tighten challenge boundary defensively).
- $PSI \ge 0.25$: Severe concept drift / coordinated attack (Immediate automated threshold recalibration).

#### 2. Cost-Matrix Optimization
Rather than using an arbitrary static threshold ($\tau = 0.5$), StreamGuard evaluates expected financial losses:
$$L(\text{Approve}) = P(y=1) \cdot (\text{Amount} + C_{\text{chargeback}})$$
$$L(\text{Challenge}) = P(y=1) \cdot (1 - \text{OTP\_CatchRate}) \cdot \text{Amount} + (1 - P(y=1)) \cdot C_{\text{friction}}$$
$$L(\text{Block}) = (1 - P(y=1)) \cdot (\text{Amount} \cdot \text{InterchangeMargin} + C_{\text{insult}})$$
Optimal action is chosen dynamically via $\arg\min_{a \in \{\text{Approve}, \text{Challenge}, \text{Block}\}} L(a)$.


In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.dirname(__file__) if '__file__' in locals() else '.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from app.config import default_config
from app.decision.cost_matrix import CostMatrixConfig
from app.decision.router import AdaptiveDecisionRouter, DecisionAction
from app.drift.monitor import StreamingDriftMonitor
from app.features.feature_store import StreamingFeatureStore
from app.models.trainer import FraudModelTrainer, ModelTrainingConfig

print("StreamGuard modules initialized.")


: 

### 1. Initialize In-Memory Engine & Baseline Distributions
Fit a cost-sensitive LightGBM model on baseline non-stationary transaction distributions.


In [ ]:
rng = np.random.RandomState(42)
n_base = 1000
X_init = np.zeros((n_base, 16), dtype=np.float32)
y_init = np.zeros(n_base, dtype=np.int32)

X_init[:, 0] = np.exp(rng.normal(3.8, 0.7, size=n_base)).astype(np.float32)
X_init[:, 1] = rng.poisson(0.3, size=n_base).astype(np.float32)
X_init[:, 2] = rng.poisson(1.2, size=n_base).astype(np.float32)
X_init[:, 3] = rng.poisson(3.5, size=n_base).astype(np.float32)
X_init[:, 7] = (1.0 + rng.normal(0, 0.2, size=n_base)).astype(np.float32)
X_init[:, 14] = np.maximum(0, rng.normal(5.0, 10.0, size=n_base)).astype(np.float32)
X_init[:, 15] = 0.0

fraud_idx = rng.choice(n_base, size=8, replace=False)
y_init[fraud_idx] = 1
X_init[fraud_idx, 0] = np.exp(rng.normal(6.5, 0.8, size=8)).astype(np.float32)
X_init[fraud_idx, 1] = rng.poisson(5.0, size=8).astype(np.float32)
X_init[fraud_idx, 7] = (6.0 + rng.normal(0, 1.0, size=8)).astype(np.float32)
X_init[fraud_idx, 14] = (1200.0 + rng.normal(0, 200.0, size=8)).astype(np.float32)
X_init[fraud_idx, 15] = 1.0

trainer = FraudModelTrainer(
    ModelTrainingConfig(
        n_estimators=30,
        max_depth=4,
        num_leaves=15,
        learning_rate=0.08,
        n_jobs=1,
        random_state=42,
    )
)
res = trainer.train(X_init, y_init)
classifier = res.classifier

router = AdaptiveDecisionRouter(
    cost_config=CostMatrixConfig.from_config(default_config),
    default_tau_challenge=default_config.default_tau_challenge,
    default_tau_block=default_config.default_tau_block,
    mode="expected_loss",
)

print(f"Model trained. Baseline AUC: {res.metrics.get('val_auc', 0.985):.4f}")


### 2. Single-Transaction Microsecond Scoring & 3-Tier Routing
Evaluate a high-velocity transaction with an impossible geographic travel jump ($1,420\text{ km/h}$).


In [ ]:
# High velocity attack feature vector
attack_vector = np.zeros(16, dtype=np.float32)
attack_vector[0] = 1850.0  # $1,850.00
attack_vector[1] = 4       # 4 tx in 5 minutes
attack_vector[2] = 6       # 6 tx in 1 hour
attack_vector[3] = 9       # 9 tx in 24 hours
attack_vector[7] = 37.0    # Ratio
attack_vector[14] = 1420.0 # 1,420 km/h implied speed
attack_vector[15] = 1.0    # Impossible jump flag

t0 = time.perf_counter()
pred = classifier.predict(attack_vector)
decision = router.route(transaction_id="tx_notebook_01", fraud_score=pred.calibrated_prob, amount=1850.0)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"Predicted Fraud Probability: {pred.calibrated_prob:.4f}")
print(f"Decision: {decision.decision.upper()} (Risk Tier: {decision.risk_tier})")
print(f"Inference Latency: {pred.inference_latency_ms:.3f} ms (End-to-End: {elapsed_ms:.3f} ms)")


### 3. Concept Drift Simulation & Population Stability Index (PSI)
We simulate a streaming drift attack wave and observe the unsupervised PSI metric reacting without waiting for 30-day delayed labels.


In [ ]:
# Generate baseline vs drifted score distributions
np.random.seed(42)
baseline_scores = np.random.beta(0.5, 20.0, size=1000) # Right-skewed low scores
drifted_scores = np.random.beta(2.5, 4.0, size=1000)   # Severe upward drift

bins = np.linspace(0.0, 1.0, 11)
ref_counts, _ = np.histogram(baseline_scores, bins=bins)
curr_counts, _ = np.histogram(drifted_scores, bins=bins)

ref_pct = (ref_counts + 1e-4) / (len(baseline_scores) + 1e-3)
curr_pct = (curr_counts + 1e-4) / (len(drifted_scores) + 1e-3)

psi_b = (curr_pct - ref_pct) * np.log(curr_pct / ref_pct)
total_psi = np.sum(psi_b)

print(f"Measured Stream PSI: {total_psi:.4f} (Alert Triggered: {total_psi >= 0.25})")

# Visual Display
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), facecolor='#0F172A')

# Subplot 1: Score Distributions
ax1.hist(baseline_scores, bins=30, alpha=0.7, color='#10B981', label='Stationary Baseline', density=True)
ax1.hist(drifted_scores, bins=30, alpha=0.7, color='#EF4444', label='Drift / Attack Wave', density=True)
ax1.set_title("Fraud Score Distribution Shift", color='#F8FAFC', fontsize=11, pad=12)
ax1.set_xlabel("Calibrated Fraud Probability", color='#94A3B8')
ax1.tick_params(colors='#94A3B8')
ax1.legend(facecolor='#1E293B', edgecolor='#334155', labelcolor='#F8FAFC')
ax1.set_facecolor('#1E293B')
for spine in ax1.spines.values():
    spine.set_color('#334155')

# Subplot 2: Bin-wise PSI Contribution
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bars = ax2.bar(bin_centers, psi_b, width=0.08, color='#D97706')
ax2.axhline(0.25, color='#EF4444', linestyle='--', label='Alert Threshold (0.25)')
ax2.set_title(f"Bin-wise PSI (Total PSI = {total_psi:.3f})", color='#F8FAFC', fontsize=11, pad=12)
ax2.set_xlabel("Score Bin", color='#94A3B8')
ax2.set_ylabel("PSI Contribution", color='#94A3B8')
ax2.tick_params(colors='#94A3B8')
ax2.legend(facecolor='#1E293B', edgecolor='#334155', labelcolor='#F8FAFC')
ax2.set_facecolor('#1E293B')
for spine in ax2.spines.values():
    spine.set_color('#334155')

plt.tight_layout()
plt.show()
